In [1]:
import warnings

import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd

from geopandas import GeoDataFrame
from pathlib import Path
from glob import glob
from cartopy.crs import Geodetic, PlateCarree
warnings.simplefilter('ignore')

# Set Parameters

In [2]:
# Parameters
year = 2018

In [3]:
# Paths
project_folder_path = Path('/gxfs_work/geomar/smomw597/2025_copepods/')
sample_folder_path = project_folder_path / 'output/sample_points/'
trajectory_path = project_folder_path / f'output/Trajectories/{year}/'
sample_points_path = sample_folder_path / f'sample_points.geojson'
voronoi_polygons_path = sample_folder_path / f'voronoi_polygons.geojson'
output_path = project_folder_path / f'output/connectivity_matrices/'
sample_points_valid_path = project_folder_path \
	/ 'output/sample_points_valid_count.geojson'

In [4]:
# Load the samplepoints and the voronoi polygons
sample_points = gpd.read_file(sample_points_valid_path).set_index('location_id_old')
sample_ids = sample_points.index.to_list()

voronoi_polygons = gpd.read_file(voronoi_polygons_path)
voronoi_polygons = voronoi_polygons.get(['geometry', 'location_id_old'])

connectivity_matrix_file_path = output_path / f'connectivity_matrix_{year}.csv'
files_list = list(output_path.glob(f'connectivity*'))
connectivity_matrix = pd.read_csv(connectivity_matrix_file_path, index_col=0)

In [5]:
trajectory_chunks = {'trajectory': 3000, 'obs': -1}
# Read and prepare the trajectory file
trajectory_file_path = sorted(trajectory_path.glob(f'*KB*'))[0]
# Make a mask that masks all days with an age under 10 days
ds_traj = (
	xr.open_zarr(trajectory_file_path, chunks=trajectory_chunks)
	.get(['age_sec', 'time'])
)


In [6]:
trajectories_age_mask = ds_traj['age_sec'].compute() > 10*24*60*60
trajectories_november_mask = (
	ds_traj.time.isel(obs=0).chunk(-1).compute() 
	< np.datetime64(f'{year}-11')
	).broadcast_like(ds_traj)
trajectories_obs_mask = (
	ds_traj.obs < (ds_traj.obs.max() - 23*4+3)
	).broadcast_like(ds_traj)
trajectories_mask = (
	trajectories_age_mask
	& trajectories_november_mask
	& trajectories_obs_mask
)

In [ ]:
# If the connectivity matrix for the given year already exists, thake that
if connectivity_matrix_file_path in files_list:
	connectivity_matrix = pd.read_csv(connectivity_matrix_file_path, index_col=0)
	print('Loaded file', connectivity_matrix_file_path)
else:
	connectivity_matrix = pd.DataFrame(index=sample_ids, columns=sample_ids)
	print('Created a new file', connectivity_matrix_file_path)
connectivity_matrix = connectivity_matrix.fillna(0).astype('int64')

Loaded file /gxfs_work/geomar/smomw597/2025_copepods/output/connectivity_matrices/connectivity_matrix_2018.csv


In [ ]:
for sample_id in sample_ids:
	cm_sum = connectivity_matrix.loc[sample_id].sum()
	print(sample_id, cm_sum)
	if cm_sum == 0:
		# Read and prepare the trajectory file
		trajectory_file_path = sorted(trajectory_path.glob(f'*{sample_id}*'))[0]
		# Only get the positions and mask
		ds_trajectories = (
			xr.open_zarr(trajectory_file_path, chunks=trajectory_chunks)
			.get(['lat','lon'])
			.where(trajectories_mask, drop=True)
		)
		try:
			lons = np.concatenate(ds_trajectories.get('lon').to_numpy())
			lats = np.concatenate(ds_trajectories.get('lat').to_numpy())
			coord_nan = np.isnan(lons)
			lons, lats = lons[~coord_nan], lats[~coord_nan]
		except:
			print('something went wrong with', sample_id)
			continue
		# Build a Gepandas Gedataframe from the dataframe
		gdf_trajectories = GeoDataFrame(
			geometry=gpd.points_from_xy(lons, lats, crs=Geodetic())
			)
		# Check for each particle which polygon contains it
		gdf_trajectories = (
			gdf_trajectories.sjoin(voronoi_polygons, how='left')
			.drop(columns=['index_right'])
		)
		# Count the particles per polygon
		polygon_count = gdf_trajectories.location_id_old.value_counts()
		# Write the counts per polygon into a connectivity matrix
		for polygon_id, count in polygon_count.items():
			connectivity_matrix.loc[[sample_id], [polygon_id]] = count
		connectivity_matrix.to_csv(connectivity_matrix_file_path)
		cm_sum = connectivity_matrix.loc[sample_id].sum()
		print('Updated file', connectivity_matrix_file_path)
	sample_points = gpd.read_file(sample_points_valid_path).set_index('location_id_old')
	sample_points.loc[[sample_id], [f'valid_particles_{year}']] = cm_sum
	sample_points.to_file(sample_points_valid_path, driver='GeoJSON')
connectivity_matrix

WH02 207818670
SK01 207819000
DB02 202052687
DB01 205055934
JB01 0
